# Linear attention, Part 2 of 3: attention in linear time

*Part 2 of 3, adapted from 2026 Round 2 Problem 1. Part 1 (u15-p05) wrote the feature maps; Part 3 (u15-p07) adds a causal mask.*

With a feature map $\phi: \mathbb{R}^{d_k} \to \mathbb{R}^{\ell}$ whose outputs are positive, the attention output for query $i$ is

$$o_i^\top = \frac{\sum_{j} \phi(q_i)^\top \phi(k_j)\; v_j^\top}{\sum_{j} \phi(q_i)^\top \phi(k_j)} \qquad (j \text{ over all } T \text{ keys}).$$

Compute it for every query **without forming any $T \times T$ matrix**, by computing $\phi(K)^\top V$ and $\phi(K)^\top \mathbf 1$ first.

```python
def linear_attention(q, k, v, feature_map):
    # Your code
    return out
```

- `q`, `k`: float tensors of shape `(B, H, T, d_k)`; `v`: shape `(B, H, T, d_v)`.
- `feature_map`: a function such as Part 1's `elu_feature_map` or `taylor_feature_map`, mapping `(..., d_k)` to `(..., l)` with positive entries. Call it on `q` and on `k`; do not apply it to `v`.
- Return `out` of shape `(B, H, T, d_v)`. Gradients must flow to `q`, `k` and `v`. Do not modify the inputs.

The grader watches every tensor your function creates: if any intermediate tensor has at least $T^2$ entries for a sequence with $T$ much larger than $\ell$ and $d_v$, the test fails. It also compares your output with the same formula computed through the $T \times T$ weight matrix.

**Example.**

```python
phi = lambda x: torch.where(x >= 0, x + 1, torch.exp(x.clamp(max=0)))
q = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])          # B = H = 1, T = 2, d_k = 2
k = torch.tensor([[[[1.0, 1.0], [-1.0, 0.0]]]])
v = torch.tensor([[[[2.0], [4.0]]]])                    # d_v = 1
linear_attention(q, k, v, phi)                          # tensor([[[[2.4488], [2.5659]]]])
```

(Here $\phi(k_0) = (2, 2)$, $\phi(k_1) = (e^{-1}, 1)$; for $q_0$, $\phi(q_0) = (2, 1)$ gives weights $6$ and $2e^{-1} + 1$, so $o_0 = (6 \cdot 2 + 1.7358 \cdot 4)/7.7358 = 2.4488$.)

**Restrictions.** Only `import torch` and `import math`. No `softmax`, `scaled_dot_product_attention` or `nn.MultiheadAttention`.

**Constraints.** $B H \le 16$, $T \le 4096$, $d_k, d_v \le 64$, $\ell \le 300$. Time limit 1 second per call on CPU.


## How to use this notebook

1. Write your solution in the next cell. Running it saves the file `problem6_submission.py`, the same way the contest's Colab notebook does.
2. Run the grader cell at the bottom. It runs the example and the hidden tests and checks the import and ban rules.
3. Edit, re-run the solution cell, then re-run the grader.

From the USAAIO Prep study site (https://bengreff.github.io/usaaio-prep/); not affiliated with or endorsed by USAAIO.

In [ ]:
%%writefile problem6_submission.py
import math
import torch


def linear_attention(q, k, v, feature_map):
    # Your code
    return out


Optional: import it the way the grader does and try the example from the statement.

In [ ]:
import importlib
import problem6_submission as submission
importlib.reload(submission)
# e.g. submission.linear_attention(...)

## Grade

In [ ]:
#@title Grader: run this cell to grade your solution { display-mode: "form" }
# The grading code is hidden on purpose (double-click to see it). It writes the site's test harness and this
# problem's grader to a temporary folder, imports your file the way the contest grader does, checks the import
# and ban rules, runs every test (including hidden ones) and prints a table.
import ast, base64, contextlib, importlib.util, io, json, os, signal, sys, time, traceback, zlib
_P = json.loads(zlib.decompress(base64.b64decode("eNq9W4132zaS/1dw6nsb0iszsmLnukrdWyd12+zl6yXude/ZPhYSIZlriqQJ0h/r9f9+vxkAJCjJTpz2zm1sEh/zhZnBzAC8HaTJYCIGzc7edjl6PhiKwVLVEk236z11WmeK2t6kuZKVkHWt8jot8qH4IKtajEUxF88mXbtIc5GZsXW6VAQjU1oXuQOcWZRp3tRKo3VnD69JOp+nsyarb9CyiwZAq+h5YIDFLQKeXCRNpuJcLpm2siqmmVo+j3UzXaZa21Eyy4orlcTpsiyqmlAdD+qimp0xCFmfDU7xMJV5rhLu1DOZYXxS1DFAJs2s7qN9C/rSMyWTA79VF/N6Ka8dsDgripKQzWWmFYlQ6TomWRRNHVPHeEStcmEoajFEhlOmvG2bFcsyU9cp5EKTiPpIpbluloxPF001YxnIRJa1SrbHo/Hz7Y/j7Q87NKHEGo3RXVeNusP7PM1Y5reDM1nlWJeoZCGf8H8/q6xUlRb1mayFFapYVDKhRiNFEfw2r4qlsNNd69ZvYXSSn+QHdrhItZgVOXHxtMnTWj/9jv58/9RCxXuZJt8/NaNBRCRe1yJRc4gA+CExMW/yGYlAC1rmhBvjLUNbLc8VntRJ3q24MEohfkPTb5E4/PvB2w9vDj9BGXVNLCkDloBpoc+Kq1xg0scmF8FlqlNQFb4QnwhcLaoGaNWlqm54Elj7UaZZU2FmJVOtBL2Kq7Q+E1IAnpYLJa6qlJZNzItKvFT5RGh5I66I3Cup0Vvki6HICapImmUpztIkUWQvZVNrkp5ZhdyKdFaUN+0LqQ/J177mmH8jADUvqfUb8arI5FS8+vCLFrJSQpPiVySrnEh5osXbsfhQFUOhC5EVUHOCmOYLLZYNhDJVJFSIqCArTNGdpSSHKfg/w1LO5QyqFxGmcTSixZVi0dD6BwazBqxMbddnFaxD6FJhwTCqKprFWXYDbcnmArSAillRkaArRUKvRTonAvHKJmr5sMZCQvn04fDwh/jHg1dH7z+KfcJODJ/ks0wCOy1DcHg9UyVpSjg5yQV+SnSZUVAoSFwlAXQxGYqlXrgxwJsXJOQ8sS300y1uQGM7IHFeBtduLhyTN8etEJmm1zoH/7DTWuYzFVwPTX90pHJdVKE3ndGquqlycR0l8MOzsyCMZmWD37zMQWgGK2ZTvGZ0h1VVVB4Uw7IHKy8jqWVVyRvQ3fGhz2SpgkVRD6GUOf2Gfu6fDKDZ8G0nA0eZhqzrBp4nIM4xPox4qqWFZYoxbhDBCodifjK4JYB3WHBtcIlbDb+jrks1g3cSt974O6DrKJtlhX6IsqGo6iLb31Hbz4fYbczjt45eWNYVSG6Jlbq+AaPzrJBEGLUzzn6Hx8zC8EcsXZnHe9gxDNjh4SbeXBez1yLAgqQa7i2tMRt0ZFkQCngK0kK/78r2+ejJk8qUPWFOc9J83gK3ikzrnWVWiBCGFRf9suKiX77iqaqCxFgQRBx2MPojpzpYiG1xFYYhwQaj6T+VUNjKxIiMb4Op9OXEjk5cygz+QQSZrBbkdhUprLjFn0n0bHEX9tZeXTQy+wKt7BuVGUpE5wlr+jBkiYII7J+V7T8Z8HJ0QDaoi6cg3Zj/c7XwVYPpj60cQNyaMflSbSHQqkxWKYYyELXMfAcDzvgWXf9W9UijUWjqLUaqYzYRXg560veuByP0VqSbskY/tUIUFnAYxRy8xXGPnAXcH62cxfrEDXrioPYpjXUuS+zk9f+va86KXD3SKQOl08prUkne2wchtsSErbdPT5CkMwiSApfQp6fPXI86AhgkSpX7R4j0PAWzBB5BfKvkrQMIe1sINUUEk/uue4LHsgRyKKZfL3f5RXK3AS/bBeP7WrFLEjvD0X1XYBFJg0QH03CTr8H0zs9snN76ial95LXtWzZxMDTeDpqd+2vl4DAEL3gp4mUDo0B0E8yRc23BncIytrbOr+jJUWJCx1fYAcQ8Z8xzClCJB8Q4iAIXMC+ZI2SsFs0SQTllaWUmZypyUSfv+mRP8IvHnWXJkONZSRMI46kZieiMNo88WCXIdNOUFIoMhnVIUxVCGVVJ7G//TMvAzGEcPQVnf9Iql3UiN0hxoIuJogA/nacUWVLI3PHyzW0q/ix27l6Iq6I6p6hesvIKWj7Eo62/tEIG8Z2EKYhP80ArCgVB1ZdK+YUvYnhXP+TeEElzLor90KSYLCCOdZ+0MxG8mxyKUL0HiIpSMCRIS00scRz+Vs7EDAhoCpg0UTdlKxzYU3CA7GJ72iTwpNTMMTjC8pQSqKaEOYn6Kp0pAggJpdihi7m1YNDsJKpZhahljniJtm+CgBkN+iJxRB1phWYwkz0hFLQvGGQkYZX01KoeUTQJCBGyy3nMYFTlvM1nVCmpN09GjFLbcMTyuy/sIoqnwk8bOpWMSRUrsoZg7OsdFhFovtu3kFZc0RQ5zXnXJMdDcT4Gtr5/ZJKHK409Tj4vCqaTBDFmMYy9ZhYDMrYgoS32HnH48abPENtRXRSsN0j+k3oSjed3iPKhZeqFzbpaJWZlgtkxINHklM/fWtHYeZxDIrFzS7zhh1Eis0QaPIdqLSiANUnmrV0nBkTW8aCBVvkC1qkSt2BdgoOVTIolpD2XUL+4HXjCZZlGS5kWMVcYelWOn6gFFgDKuuJByk7jLM0SsroZVUaQMKdkCzTyb5/ev+OyFlmOrhNQyLZa3tRnaNl+LRwe8Z0tccRJWn0vvutwoBPvsMp/cVv9/VrKD2Pz0n8s7HWdpdO2yfxBS9TUadY1F+3jP3TRFQlKWZ/503W6yGU3Td/olfKCe6mwL0zl7LxtgRbnVCwwtQY9S8Fmrlp3h4bztN7OFIaJneh5tGNjOj2hOtGS6n9DdinILuzm45xfKVNIeXGSOxzRPM2g0e41OBmA7KJSlAHaSgsi0Wjrl/w8pxqOLjKqqBRcAQDCtKRVi7ZspPjz4cdDWI6VRfQBf4M4pjJYHIcRHBdNR1RXygqUneSQSkSDI2irqupgBGNA+EBgQgDcVPviWpD4Bnv1hZyIw93RmCthb349+O9P8cuDd+8OfwAFtyeD2FYi45iYAWNuPc2rulYz+4Q43zxR+Q+0ngzuTvI3799/iN+9/+HwE8AF0JXoxwK2Sg+/QnGVeTzQN/ms7XiDGPIVYJi3T8p7+QFBZvf2k8ppfy6qw+sy9EssR6YW87kqSyyR6S0D0rFmCX+DbUu1FsupogPkVyMospnFszM1O9eBrma0xLXs77cfjb1TNEy1ZsixODfCWeoFBPMCbpEiQFPjY7PR9ZN2dTgYsqVE2qjqKjVlxd4WZTah49NN0WxdKQR2LCVoCfJrENqPQz/dID+/5jiUinJqPT48tmRPoC1cFbbUT8hNap5uU+TClM+RKnM9OC/uJvSM0Uh+7iyBtrJNWwLkFcExw05Wyt0ko+NTS+hUtu28T9Zuy+FyaNxoBua1T2EPsamV2nZScYSMsN00oeI0IrHKBAoGMCCY8QFlNBR0p1Oq8IOdc47dVBaRYlsEXCX3ZL6U9bLJ/JY5V0fgqsk3Q/hXMjsPaDFWtm4vTKfhRp9NarCaVbShLA2MmNzJ+gZmSyqIlwPJgygoX1JpwbxGwAZBUuQYLBF5wh1RMkfAl0ysWYhwsnlv9NYikklicYTrg7u18cZFusxSWm7Ceby9c7phYrd4UVMmFHVvmOsnidnDYvwRXm+VGywo1orlaAvvYL8zqA2y5BlWkHj+HVJclSCgrUjBE11PAps0YGXqfcL7qqnM52OFfuBsZ5X3FYVgOsjQvgjqO88jPwQwTe7navOgTRi7zWoVJxt+JMtS5UngKj5m0snAuDzj3v+jL69NWHgXfJnm7+0udtAsDjTtQaGpraxMiAo77q2s6SxvTaeND3okcSDMU0gPJAWJFtS97t9FAcZKjMpTKxsEjYv+UVB6yg428PCE4Wbv8mfaUSAYB+r2yVA8MUCcRZEBPgE+5L2LJ1SKvdtcT9xEP5UsNpCvHe0dsG/aE8SpzDXJdnZmdwgOBbsNgsvPSiKJ/Tv/DV+YbDS7kjd625zPIpNBXJi2gEgpLbQiz26GDicyl26rCYwWmCKSDaUo6LgRmVogNb+isEnwyCiKQm8zQ1AmkoIWxQA+Szm/NKsQeJvmn3hz9HZhQ67dfKGG/xKBt5f+SfTiQk+FgOHRqgMKtLDyIfiBDjfoDiCHv2eF88LhaI9ku1UG5X3mY2NEcVGaeLJfbaPhxshWTO9LGKaMEnrxV+FgU8rq2Y0ITORkMRyPTn3l3sD+Y0TQYX2IeXZuayxz69cxbMIkVvYeo2gmMyalrtSZyjmJbWNHnvXH8m85u3sgU59XKcBkNzFHsgFAINGNKZvqR/Tvc7XNdCZKz6qUswpRQlVh3wthT6J5QOqOrylhg7kqGqf5sL5fYZpSWcXlrRHyZjzXcT0NVBTHbQdSPhtsEnBEmnPe1OeEByCwUnNKP5WL+Sjza1mwUekVlUyoYEVelmuTVunwm8IxL2rvNdFSsMLQaD5O68i3wqQTEz4hUd75CEX/d7eM9Y7GO2EvEYkHbS4GT7ua5VImK6vF5XEbIjpW7h05PvVSYispCsD3RTvkmQvXofdop4JDlBUy0UHARDwltSGb4FLEgOBhz6JKRuCcHdIngmiJ8Qc4aXA1cV+s24VJGKnp+JTeKdXT3aspzBiVtZk0aSI1vCuwBD784w7aKZFzTyraGjpFsGSC+thQdcq7Bb2xlZz64axL6Vw8q7nuuIbW9xJctQhYnHQ5RAdmuL/LGy1xJc85iE6L6FNN5ZPX753wTJEnMn8C+/bp9U8Hbz6+Hdos3Q6FO4u5uOtOgT1f1r+zRLIcjxwt/eyYL8F0xSosZ5JWCsZnFiMApathlqNR1SmhqByZr49evz38GH88PHgzbKlbCXN0qUh7+qWwiFpjqs2Yyg5VJ/nIxOPIuzJmVKMrzGGZPFe1gq+ZrqOzsBghoQ7o1+pEWIwZp4+pmxOGU6P46yyxDakqohpQbKaR5wm/QnCjfm3Cll16JQmriNY4To03MzySB2avxh63LopzPlMQt249JtGIKsDOdVksL6VWbYXI1EC6ktjLN4ej0U5HwKPYuI9iezDEZNJJDN0QK+xC2SDogf2oNes10JsE5fwK4YU+k0bxSXyABHx3NBpNTr/akPmaY7+Ow03+/Q4zxp0VuxNN8DI0XeHnF7eTVVIoEzSZy3bYXQjEnZ+3P5KFxeNs8mRgLv2RFbZbRnsPsOXb3iP8vO0tPONb3GdLBpodZa/72cKWy/LMEPIM7uqgC+Kt26O9hoKGAInK3Dh2eiLffikr7XBEaa2WOuDbM3l/V2Cv6s796QBFgoxgHvoFL3gJOoTkkIQwrgTPvB2T9lcN/JZRCopW+BZB7lhbcbjkntO8Uf76zcwGa/zhhIEc701O4b/5XBjU2iq1YPd4MrCg3WAPG/VSNZjSnk1pivW35tCHmkbR6O4xZ2LrFx++Ytf5vTuPPZxb98pf5dLsIhx3kiNLpcB7/e7GmgfvJpvYw9g4O+usyBfuXGbVZ4sgzc0tM47mh5RAzOmetj0ZDH0nYJHz4cZKTft3MtzSTOH1hrsqj9pL/iha/HsGfF6RfOEe0sFqFZzgVdDiJLjvoHYonm3a3mw4e+oSsqp1bI/eitoJJlbdp9Qx6GLXh6JSdirTosiCNcJMH4Gqe5pL4OoeuHaGS1Du3VL4WvNcuITHeLc4pvyGDsom7niA0h1zsttuFnSma8tndJ2/O8E0F4/4sM4Uf2jr0FFsjmvjJNUlV5Hs+CMa84NtfAsVuP+gb8iH4kNzXXUo7JVA75rO0N4oIbbaO+jYNaz3V9eSLvRDMv4h2xtzS9JchQrWyFk9EJsVVWKOu/JmOaXTTxMkmPscJn229y5rBsl3f1gStorBGTd/4qD6B2F8lheTk4jjQKus50Z1U5IKR+0AT4lpbOSw7ouRu8liADLuVu4W9JALSu0lQ7oqsU8XX82tiX1K2cJ+sYocHaZ0V0XsDQvyZLd3foBCGlmQRgY0rV87RstQBHwt092yMxn5MXpOV/eNlbkPX1vbIAm6WOs3we3SxW6VBesBVb+eUp6lscqa7nZj71IclwSCa/E9hD0U13QDytGmrsvgOoJqLcsA6PeRuoV9sLW8yaBpLWQ6y7k2F9eoYmEaLxpJ7cH1cRRFQzEZchZ9KraEbaFXtJ+G0TzjT1aC7XG4gVREfsGxecQMHWw5VBPgGhLdCenA/nXEf0Ni5ylo2tpCqDDeGxpKnopgDNxJaNr3MDNJl/vbOx5vlZpDKrRQF9CjobgcEruOyzka5+dU/jhLg4sInpTirzDkQcF512CG/0rqdiH+ijlRXclcl3S7ens8FMApHv75RgQvh+LnoTjC/32ZBL8C5GWHDIz9GulmGRhuQLdSJT3b64Etb/C9wZb1OBtuxBgB06i8HbZwx/r7prc95ocVL2VOFxE13+UncP5tfKqDmBTDFyTMRU5Ve43cWKR/RxEzotUPtvogXJ2CP0RgS7z0L1Yz/LsJgS6b7mKQudjOw+9Z4s3Tu68GdruvBvYeqGH6TppjPcvqBcWoLELjUYNj/OxEsDwEs9DE4xE9o+EUPxb++b1zdsyc7RZAN+ly06SxGb/bG3nPGpHLMNdJ7E4TrnKXNfGU3DCte8ehB441DWr+bMjfqn1r1G1/J7y3a7zetWe7noU9snbgqA7f/DJs7SMkF+OmYWwSkwj21uk2PiueK4llUwgQyofIhx0BKH1buGsJ2Q3v7dpb73puu55b8g124uCIn4SlA/69FEFG3ybthOtEwziO4ukNfiFCqdJrn+QjTNoZjXethpk1pNixo+SoE/G/92n0er69t+cvVlU4XeoHGSFF1lekBmtb7CYjDlZVzL9ryGDaLe87MLZFVMAgicXbIxikKUC4ezizCsKjK8kuPmECb3tw7lxIQ4YrMiUB+ggbgwFJGO5e8PlHg3yG/Pd/hv9zJP7L3Ez9csfBvPRp/azz6K0wRaMpXZrtuYt2MQNvZZ53GqfDdgNA8nrRIHvVDCoO2O13EXqwQw4DAHbGYe/K7KOXiY56eriGtvLC5E/MtWXm3a8SUSZiLiAr+60iHvnDPiGbmi4odxesOdjasv5Lmpu27OP26JfZ5E0nV5yf74Yh73xhRMczCOUSF1JewNTO8e+SSkBBvfb5xf1C4zzEycFJLPAWv4XcCemPoZmwex8xuSrJMRFzwa7jgmspwbmh4GRwbt4vDTUng0u8n274oiciJt1pp4m65r2lywvh3qDnkny7uCUC+p8btR/ZRWb5iVD72IfHyhI+TRyMe76+WzEF/u5zsxmwFeyy/u+O/gIgz50l7Dxz7mtz9+7D3Xs9m7C3+Xn9Ph+LeAbixSMriLj4xT5JzKRWvdBkcHf3v2VQSQw=")).decode())
_dir = os.path.join("/tmp", "usaaio_grader_" + _P["id"])
os.makedirs(_dir, exist_ok=True)
for _n, _s in _P["files"].items():
    if _n.endswith("::b64"):
        with open(os.path.join(_dir, _n[:-5]), "wb") as _f:
            _f.write(base64.b64decode(_s))
        continue
    with open(os.path.join(_dir, _n), "w", encoding="utf-8") as _f:
        _f.write(_s)
if _dir not in sys.path:
    sys.path.insert(0, _dir)
for _m in ("harness", "usaaio_grade", "grader", _P["meta"].get("module_name", "submission")):
    sys.modules.pop(_m, None)
import harness as _harness
_harness.SPEED_FACTOR = 1.0   # Colab is the reference machine: time limits apply as stated
import usaaio_grade as _G

def _usaaio_grade():
    meta = _P["meta"]
    mod = meta.get("module_name", "submission")
    path = os.path.abspath(mod + ".py")
    rows, static = [], []
    if not os.path.exists(path):
        print(f"{mod}.py not found: run the %%writefile cell above first.")
        return
    src = open(path, encoding="utf-8").read()
    static = _G.static_checks(src, meta)
    per = float(meta.get("test_timeout_s", 20))
    def alarm(*_):
        raise TimeoutError()
    use_alarm = hasattr(signal, "SIGALRM")
    if use_alarm:
        signal.signal(signal.SIGALRM, alarm)
    def limit(s):
        if use_alarm:
            signal.setitimer(signal.ITIMER_REAL, s)
    buf = io.StringIO()
    error = None
    if any(not s["ok"] and s["msg"].startswith("syntax") for s in static):
        error = "syntax error"
    else:
        try:
            with contextlib.redirect_stdout(buf):
                limit(per)
                spec = importlib.util.spec_from_file_location(mod, path)
                sub = importlib.util.module_from_spec(spec)
                sys.modules[spec.name] = sub
                spec.loader.exec_module(sub)
                limit(0)
        except TimeoutError:
            error = f"importing your file took over {per:.0f} s"
        except BaseException as e:
            limit(0)
            error = "your file failed to import: " + _G.friendly_error(e, path)
        entry = meta.get("entry")
        if not error and entry and not hasattr(sub, entry):
            error = f"your file does not define {entry}"
    if not error:
        gspec = importlib.util.spec_from_file_location("grader", os.path.join(_dir, "grader.py"))
        grader = importlib.util.module_from_spec(gspec)
        sys.modules["grader"] = grader
        gspec.loader.exec_module(grader)
        visible = set(getattr(grader, "EXAMPLES", []))
        for name, fn in [(n, f) for n, f in vars(grader).items() if n.startswith("test_") and callable(f)]:
            rec = {"name": name[5:].replace("_", " ") + (" (example)" if name in visible else ""), "passed": False, "msg": ""}
            t0 = time.perf_counter()
            try:
                with contextlib.redirect_stdout(buf):
                    limit(per)
                    fn(sub)
                    limit(0)
                rec["passed"] = True
            except TimeoutError:
                rec["msg"] = f"took longer than {per:.0f} s (infinite loop, or far too slow)"
            except _harness.Fail as e:
                limit(0)
                rec["msg"] = str(e)
            except BaseException as e:
                limit(0)
                rec["msg"] = "your code raised " + _G.friendly_error(e, path)
            rec["seconds"] = time.perf_counter() - t0
            rows.append(rec)
    print("Rule checks")
    for s in static:
        print(f"  {'PASS' if s['ok'] else 'FAIL'}  {s['msg']}")
    if error:
        print("\n" + error)
    if rows:
        w = max(len(r["name"]) for r in rows)
        print("\nTests")
        for r in rows:
            print(f"  {'PASS' if r['passed'] else 'FAIL'}  {r['name']:<{w}}  {r['seconds']:6.2f} s  {r['msg']}")
    out = buf.getvalue()
    if out.strip():
        print("\nPrinted output (last 2000 characters)\n" + out[-2000:])
    n_ok = sum(r["passed"] for r in rows)
    ok = not error and rows and n_ok == len(rows) and all(s["ok"] for s in static)
    print("\n" + (f"Accepted: all {len(rows)} tests passed and every rule is met." if ok else
                  f"Not accepted: {n_ok} of {len(rows)} tests passed" + ("" if all(s["ok"] for s in static) else ", and a rule check failed") + "."))

_usaaio_grade()
